In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

# Bild einlesen (falls noch nicht geschehen)
image_path = 'ditto.png'  # Passe den Pfad an
img = Image.open(image_path).convert('L')
img_array = np.array(img)

# Bild anzeigen
plt.figure(figsize=(4, 4))
plt.imshow(img_array, cmap='gray')
plt.title("Original BMP")
plt.axis('off')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from scipy.fft import fft, ifft
from scipy.ndimage import center_of_mass
from skimage import measure

In [ ]:
# === Bild einlesen und binarisieren ===
#image_path = 'ditto.bmp'  # Pfad zum Bild
img = Image.open(image_path).convert('L')
img = np.array(img)
binary = (img < 128).astype(np.uint8)  # Schwarz=1, Weiß=0

In [ ]:
M = min(binary.shape)
M

In [ ]:
# === Mittelpunkt der Figur berechnen ===
cy, cx = center_of_mass(binary)

In [ ]:
# === Kontur extrahieren ===
contours = measure.find_contours(binary, level=0.5)
# Wir nehmen die längste Kontur
contour = max(contours, key=len)

In [ ]:
# === Polarkoordinaten berechnen (r, θ) ===
x = contour[:, 1] - cx
y = contour[:, 0] - cy
theta = np.arctan2(y, x)
r = np.hypot(x, y)

In [ ]:
# Sortieren nach θ für sinnvolle Fourier-Analyse
theta_sorted_idx = np.argsort(theta)
theta = theta[theta_sorted_idx]
r = r[theta_sorted_idx]

In [ ]:
# === Fourier-Analyse ===
N = len(r)
r_fft = fft(r)
r_reconstructed = ifft(r_fft).real

In [ ]:
# === Plot ===
plt.figure(figsize=(10, 5))

# Originale Kurve (in Polarkoordinaten)
plt.subplot(1, 2, 1, polar=True)
plt.plot(theta, r, label='Original')
plt.title('Original r(θ)')

# Rekonstruierte Kurve (nur erste n Fourier-Koeffizienten)
n = 15  # Anzahl der Koeffizienten
r_fft_filtered = np.zeros_like(r_fft)
r_fft_filtered[:n] = r_fft[:n]
r_fft_filtered[-n+1:] = r_fft[-n+1:]
r_smooth = ifft(r_fft_filtered).real
#r_fft_filtered

plt.subplot(1, 2, 2, polar=True)
plt.plot(theta, r_smooth, label='Fourier Fit (n=10)')
plt.title('Fourier-Fit r(θ)')

plt.tight_layout()
plt.show()

In [ ]:
plt.subplot(1, 2, 2)
plt.plot(theta, r_smooth, label='Fourier Fit (n=10)')
plt.title('Fourier-Fit r(θ)')

plt.tight_layout()
plt.show()

In [ ]:
c = r_fft / len(r_fft)  # Normalize
a0 = c[0].real / M / 1.2
ak = 2 * c[1:n].real / M / 1.2
bk = -2 * c[1:n].imag / M / 1.2

print(f"a0 = {a0}")
for k in range(1, n):
    print(f"a{k} = {ak[k-1]:.4f}, b{k} = {bk[k-1]:.4f}")

In [ ]:
from numpy import linspace, pi, cos, sin, exp
x = linspace(-pi,pi,100)
y = a0 + sum([ak[k-1] * cos(k*x) + bk[k-1] * sin(k*x) for k in range(n-1)]).real
plt.subplot(1, 2, 2)
plt.plot(x, y, label='asdf')
plt.title('Fourier-Fit r(θ)')

plt.tight_layout()
plt.show()

In [ ]:
from ngsolve import *
from ngsolve.webgui import Draw
mesh = Mesh(unit_square.GenerateMesh(maxh=0.1))
lset = x
X = CF((0.5-x,y-0.5))
theta = IfPos(X[0],atan(X[1]/X[0]),atan(X[1]/X[0])+pi)

In [ ]:
R = a0 + sum([ak[k-1] * cos(k*theta) + bk[k-1] * sin(k*theta) for k in range(n-1)]).real

In [ ]:
d= sqrt(X[0]**2+X[1]**2) - R

In [ ]:
Draw(d,mesh,"x",subdivision=4,order=5,min=0,max=0,autoscale=False)
# Bild einlesen (falls noch nicht geschehen)
image_path = 'ditto.png'  # Passe den Pfad an
img = Image.open(image_path).convert('L')
img_array = np.array(img)

# Bild anzeigen
plt.figure(figsize=(4, 4))
plt.imshow(img_array, cmap='gray')
plt.title("Original picture curve")
plt.axis('off')
plt.show()